# FCP LoRA Training

In [ ]:
import torch; print('GPU:', torch.cuda.is_available())

In [ ]:
!pip install unsloth xformers trl peft accelerate -q 2>&1 | tail -2

In [ ]:
from google.colab import files; files.upload()

In [ ]:
import json
from datasets import Dataset

dataset = json.load(open('lora_dataset.json', 'r', encoding='utf-8'))
dataset = Dataset.from_list(dataset)
print('Dataset:', len(dataset))

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-3B', max_seq_length=512,
    dtype=torch.float16, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'])
print('Params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

def formatting_func(sample):
    return [f"### Инструкция: {sample['instruction']}\n\n### Ответ: {sample['output']}"]

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset, formatting_func=formatting_func,
    max_seq_length=512, packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, max_steps=100, learning_rate=2e-4, fp16=True,
        logging_steps=10, output_dir='/content/fcp_adapter', report_to='none'))

print('Training...')
trainer.train()

In [ ]:
trainer.save_model('/content/fcp_adapter'); print('Done!')

In [ ]:
import shutil
shutil.make_archive('/content/fcp_adapter', 'zip', '/content/fcp_adapter')
from google.colab import files; files.download('/content/fcp_adapter.zip')